# Extra: why OGC API?

Geonorge has offered the same data for years, as files to download and through WMS and WFS services. So what's different about an OGC API? This notebook shows it, with the API from **part 2** (run it to step 6 first) and Geonorge's own services for the same data, Kartverket's municipalities:

1. **One question, four ways:** "which municipality is this address in?"
2. **A picture, or data?** WMS sends a finished picture. Vector tiles send data, so you decide how it looks.
3. **Fresh data:** change the database, and the API follows straight away.
4. **Same code, Kartverket's API**

Everything here calls Geonorge's services over the internet, so a cell can fail if one of them is down.

In [ ]:
import io
import json
import urllib.parse
import zipfile

import shapely
from pyproj import Transformer
from workshop import *

## 1. One question, four ways

The question: **which municipality is Kartverket's address in?** First, the address's coordinates, from Geonorge's address API:

In [ ]:
ADDRESS = "Kartverksveien 21, Hønefoss"   # ✏️ or any other address

hit = fetch_json("https://ws.geonorge.no/adresser/v1/sok", {"sok": ADDRESS, "treffPerSide": 1})["adresser"][0]
lon, lat = hit["representasjonspunkt"]["lon"], hit["representasjonspunkt"]["lat"]
x, y = Transformer.from_crs(4258, 25833, always_xy=True).transform(lon, lat)   # the same point in UTM 33
print(f"{hit['adressetekst']}, {hit['poststed']}: lon {lon:.5f}, lat {lat:.5f} (UTM 33: {x:.0f}, {y:.0f})")

### a) Download a file from Geonorge

The way most people get Kartverket's data today. Geonorge's download API takes an order (dataset, area, format and projection) and hands back a link to a zip file. You can't order "the municipality at this point", only whole areas, so you get every municipality in Norway and search the file yourself.

In [ ]:
order = {"orderLines": [{
    "metadataUuid": "041f1e6e-bdbc-4091-b48f-8a5990f3cc5b",   # the dataset "Kommuner" in Geonorge
    "areas": [{"code": "0000", "type": "landsdekkende", "name": "Hele landet"}],
    "formats": [{"name": "GeoJSON "}],
    "projections": [{"code": "25833"}],
}]}
receipt = json.loads(fetch("https://nedlasting.geonorge.no/api/order", "File download",
                           data=json.dumps(order).encode(), content_type="application/json"))
zipped = fetch(receipt["files"][0]["downloadUrl"], "File download")

# Unzip it and search every feature in it, on your own machine
with zipfile.ZipFile(io.BytesIO(zipped)) as z:
    data = json.loads(z.read(z.namelist()[0]))
point = shapely.Point(x, y)
found = [f["properties"] for f in data["features"] if shapely.geometry.shape(f["geometry"]).intersects(point)]
print(f"\nSearched {len(data['features'])} features:", [(f.get("kommunenavn"), f.get("kommunenummer")) for f in found])
extracted = data["features"][0]["properties"]["datauttaksdato"]
print(f"The file was extracted from the database on {extracted[:4]}-{extracted[4:6]}-{extracted[6:8]}")

### b) WMS

A WMS draws a **picture**. It can tell you what's at a point, but you ask about a *pixel*: "in a 500 × 500 picture of this area, what's at pixel (250, 250)?" The answer is text, meant for a popup.

In [ ]:
from IPython.display import Image, display

WMS = "https://wms.geonorge.no/skwms1/wms.adm_enheter2"
area = {"service": "WMS", "version": "1.3.0", "layers": "kommuner", "styles": "", "crs": "EPSG:25833",
        "bbox": f"{x - 30000},{y - 30000},{x + 30000},{y + 30000}", "width": 500, "height": 500}

picture = fetch(WMS + "?" + urllib.parse.urlencode({**area, "request": "GetMap", "format": "image/png"}), "WMS")
display(Image(picture))

info = fetch(WMS + "?" + urllib.parse.urlencode({**area, "request": "GetFeatureInfo", "query_layers": "kommuner",
                                                 "info_format": "text/plain", "i": 250, "j": 250}), "WMS")
print(info.decode()[:600])

### c) WFS

A WFS sends **data**, as GML (XML). A spatial question is written in XML too, and sent as a POST:

In [ ]:
request = f"""<wfs:GetFeature service="WFS" version="2.0.0"
    xmlns:wfs="http://www.opengis.net/wfs/2.0" xmlns:fes="http://www.opengis.net/fes/2.0"
    xmlns:gml="http://www.opengis.net/gml/3.2"
    xmlns:app="https://skjema.geonorge.no/SOSI/produktspesifikasjon/AdmEnheter/20240101">
  <wfs:Query typeNames="app:Kommune">
    <fes:Filter>
      <fes:Intersects>
        <fes:ValueReference>app:område</fes:ValueReference>
        <gml:Point gml:id="p1" srsName="urn:ogc:def:crs:EPSG::4258"><gml:pos>{lat} {lon}</gml:pos></gml:Point>
      </fes:Intersects>
    </fes:Filter>
  </wfs:Query>
</wfs:GetFeature>"""

gml = fetch("https://wfs.geonorge.no/skwms1/wfs.administrative_enheter", "WFS",
            data=request.encode(), content_type="application/xml").decode()
print(gml[:800], "…")

Notice `<gml:pos>{lat} {lon}</gml:pos>`: EPSG:4258 puts latitude first, so the coordinates go in the "wrong" order. Getting that wrong is a classic WFS mistake. (A WFS can leave out the geometry too, with more XML: `wfs:PropertyName`.)

### d) Your OGC API

One URL, which works in a browser, QGIS, Excel or a script alike:

In [ ]:
url = "http://localhost:3000/collections/kommuner/items?" + urllib.parse.urlencode({
    "filter": f"S_INTERSECTS(geometry, POINT({lon} {lat}))",
    "skipGeometry": "true",
    "f": "json",
})
print(urllib.parse.unquote_plus(url), "\n")
answer = json.loads(fetch(url, "OGC API"))
print([(f["properties"]["navn"], f["id"]) for f in answer["features"]])

### What did each answer cost?

In [ ]:
compare()

All four found the same municipality, but:

- **The file** answered a question about one municipality by sending all of them, and it's a snapshot: look at the date it was extracted. Every update means a new download.
- **The WMS** is a picture. Fine for a background map, but you can't calculate with it or change how it looks.
- **The WFS** has the data, but in XML, with requests written in XML and a schema of its own to learn.
- **The OGC API** answered with one readable URL and a few hundred bytes of JSON, straight from the database.

## 2. A picture, or data?

The WMS picture above looks the way the *server* decided. Vector tiles carry the data itself (geometries and attributes), so the *client* decides. Below, your API's tiles are drawn by MapLibre, an open-source map library for the web, with a **style**: a list of rules in JSON, the same format Kartverket uses for its own styles.

Edit the style and re-run the cell. Some things to try: change the colours, the opacity or the line width, or remove the labels. Click a municipality to see what the tile holds about it.

> Like QGIS, the map loads the tiles from your public address, so port 3000 must be public: **Ports** tab → right-click port **3000** → **Port Visibility → Public**.

In [ ]:
STYLE = [
    {   # Fill each municipality with a colour per county, like the QGIS step in part 2
        "id": "kommuner-fill", "type": "fill", "source": "kommuner", "source-layer": "kommuner",
        "paint": {
            "fill-color": ["to-color", ["concat", "hsl(", ["%", ["*", ["to-number", ["get", "fylkesnummer"]], 37], 360], ", 55%, 70%)"]],
            "fill-opacity": 0.75,
        },
    },
    {   # White borders
        "id": "kommuner-line", "type": "line", "source": "kommuner", "source-layer": "kommuner",
        "paint": {"line-color": "white", "line-width": 0.8},
    },
    {   # Names, from zoom 7
        "id": "kommuner-label", "type": "symbol", "source": "kommuner", "source-layer": "kommuner", "minzoom": 7,
        "layout": {"text-field": ["get", "navn"], "text-font": ["Open Sans Semibold"], "text-size": 11},
    },
]

show_map(STYLE, center=(10.5, 60.5), zoom=6)

Same tiles, any look. The frontend draws them green, QGIS draws them the way you set it up in part 2, and this map the way `STYLE` says. Publishing a recommended style *next to* the data is what OGC API – Styles is for.

## 3. Fresh data: the API reads the database live

In part 1 the GeoPackage was copied into the backend when it started, so a change to the data meant restarting the API. With PostGIS, pygeoapi asks the database on every request. Change the database, and the API shows the change straight away, in the web pages, the GeoJSON, the tiles and the downloads. Compare that with the file in section 1.

Kartverket's address is in Hønefoss, which is missing from `byer`. Add it with plain SQL, the way a data platform updates a table. The SQL finds the municipality itself, with the same spatial test as above.

In [ ]:
TOWN = "Hønefoss"   # ✏️

run_sql(f"""
    INSERT INTO adm.byer (id, navn, kommunenummer, kommune, fylke, geom)
    SELECT (SELECT max(id) + 1 FROM adm.byer), '{TOWN}', k.kommunenummer, k.navn, k.fylke, p.geom
    FROM (SELECT ST_Transform(ST_SetSRID(ST_MakePoint({lon}, {lat}), 4258), 25833) AS geom) p
    JOIN adm.kommuner k ON ST_Intersects(k.geom, p.geom)
    RETURNING id, navn, kommune, fylke
""")

Ask the API again, straight away, with no restart:

In [ ]:
items = get_json("/collections/byer/items", {"filter": f"navn = '{TOWN}'", "filter-lang": "cql2-text"})
for f in items["features"]:
    print(f["id"], f["properties"], f["geometry"])

And on the map: the towns from the API's tiles, with the new one among them.

In [ ]:
show_map([
    {"id": "kommuner", "type": "line", "source": "kommuner", "source-layer": "kommuner", "paint": {"line-color": "#888"}},
    {"id": "byer", "type": "circle", "source": "byer", "source-layer": "byer",
     "paint": {"circle-radius": 7, "circle-color": "#c0392b", "circle-stroke-color": "white", "circle-stroke-width": 2}},
    {"id": "byer-navn", "type": "symbol", "source": "byer", "source-layer": "byer",
     "layout": {"text-field": ["get", "navn"], "text-font": ["Open Sans Semibold"], "text-offset": [0, 1.3], "text-anchor": "top"}},
], center=(lon, lat), zoom=8)

Reload the `byer` collection page in the frontend too: the new town is on its map and in its table.

To remove it again, remove the `#` and run the cell. It's commented out so that **Run All** doesn't remove it straight after adding it.

In [ ]:
# run_sql(f"DELETE FROM adm.byer WHERE navn = '{TOWN}' RETURNING id, navn")

## 4. Same code, Kartverket's API

Because OGC API is a standard, code written for your API works on any OGC API. Here are some of the same requests to Kartverket's own demo service, which runs the same software as yours. It's a test environment, so it may now and then be down.

In [ ]:
KARTVERKET = "https://datadeling-ogcapi.atgcp1-dev.kartverket.cloud"

for c in fetch_json(f"{KARTVERKET}/collections", {"f": "json"})["collections"]:
    print(c["id"], "-", c["title"])

# Oslo, by its municipality number
oslo = fetch_json(f"{KARTVERKET}/collections/kommuner/items/0301", {"f": "json"})
print("\n", oslo["id"], oslo["properties"])